# KPI 5 — Customer Lifetime Value (CLV)

**requirements.md (Customer Intelligence, KPI 5):** "Customer Lifetime Value (CLV): Total spend per buyer company; segment into cohorts."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_5_customer_lifetime_value` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per **buying company** (`company_id`) — the 390 companies with ≥ 1 valid-date order line (KPI 2 convention; all are `company_type = 'buyer'`). Company attributes come from the current dim version (`__END_AT IS NULL`).
* **CLV bases (KPI 1 convention):** `clv_gross` = `SUM(line_total)` over ALL statuses with `order_date_valid = true`; `clv_net` = same excluding `status = 'cancelled'`. Totals reconcile exactly with KPI 1/2/3/4 (net 28,333,498.48 / gross 30,723,269.53).
* **Acquisition cohort:** month of the company's **first order** (any status — the acquisition moment is when the order was placed) on a valid date: `cohort_year_month` (int YYYYMM) plus a coarser `cohort_quarter` label (`YYYY-Qn`) for segmentation. 26 cohorts, 202410–202611.
* **Spend pattern:** `order_count` = distinct valid-date orders (any status); `active_months` = distinct year-months with ≥ 1 order; `lifetime_months` = inclusive year-month span first→last order (`(YEAR(last)-YEAR(first))*12 + (MONTH(last)-MONTH(first)) + 1` — **not** `DATEDIFF(month)`, which truncates to complete months and undercounts by one whenever the last order's day-of-month is earlier than the first's); `avg_monthly_net_spend` = `ROUND(clv_net / active_months, 2)` — spend **per active month**, not per calendar month.
* Customer-level AOV / repeat-purchase rate is KPI 6; Pareto top-20% is KPI 2 — not duplicated here.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_5_customer_lifetime_value
TBLPROPERTIES (
  'comment' = 'KPI 5 - Customer Lifetime Value (requirements.md: total spend per buyer company; segment into cohorts). Grain: one row per buying company_id (390 companies with >= 1 valid-date order line, KPI 2 convention; all company_type = buyer). clv_gross = SUM(line_total) ALL statuses, order_date_valid = true; clv_net = same excluding status = ''cancelled'' (KPI 1 bases). Acquisition cohort = year-month (int YYYYMM) and quarter (YYYY-Qn) of the company''s first valid-date order of any status; 26 cohorts 202410-202611. order_count = COUNT(DISTINCT order_id) valid-date orders; active_months = COUNT(DISTINCT year-month of order_date); lifetime_months = (YEAR(last)-YEAR(first))*12 + (MONTH(last)-MONTH(first)) + 1 (inclusive year-month span; DATEDIFF(month) would truncate to complete months and undercount when day(last) < day(first)); avg_monthly_net_spend = ROUND(clv_net / active_months, 2) - net spend per ACTIVE month, not calendar month. Company name/country/industry from current SCD2 version (__END_AT IS NULL); aggregation keyed on company_id (business key) via the fact''s as-of company_sk so future re-versioning cannot split a buyer.'
)
AS
WITH lines AS (
  SELECT
    f.company_sk,
    f.order_id,
    f.order_date,
    f.status,
    f.line_total
  FROM b2b_ecommerce.gold.fact_order_items f
  WHERE f.order_date_valid = true
),
mapped AS (
  -- business key via the fact's as-of company_sk (0 orders span > 1 company_sk, verified KPI 2)
  SELECT
    d.company_id,
    l.order_id,
    l.order_date,
    l.status,
    l.line_total
  FROM lines l
  JOIN b2b_ecommerce.gold.dim_companies d
    ON d.company_sk = l.company_sk
),
per_company AS (
  SELECT
    company_id,
    MIN(order_date)                                        AS first_order_date,
    MAX(order_date)                                        AS last_order_date,
    COUNT(DISTINCT order_id)                               AS order_count,
    COUNT(DISTINCT DATE_FORMAT(order_date, 'yyyyMM'))      AS active_months,
    SUM(line_total)                                        AS clv_gross,
    SUM(CASE WHEN status <> 'cancelled'
             THEN line_total ELSE 0 END)                    AS clv_net
  FROM mapped
  GROUP BY company_id
)
SELECT
  c.company_id,
  c.name                                              AS company_name,
  c.country,
  c.industry,
  p.first_order_date,
  CAST(DATE_FORMAT(p.first_order_date, 'yyyyMM') AS INT)                     AS cohort_year_month,
  CONCAT(DATE_FORMAT(p.first_order_date, 'yyyy'), '-Q', QUARTER(p.first_order_date)) AS cohort_quarter,
  p.last_order_date,
  p.active_months,
  ((YEAR(p.last_order_date) - YEAR(p.first_order_date)) * 12
     + (MONTH(p.last_order_date) - MONTH(p.first_order_date)) + 1)          AS lifetime_months,
  p.order_count,
  p.clv_gross,
  p.clv_net,
  ROUND(p.clv_net / NULLIF(p.active_months, 0), 2)                            AS avg_monthly_net_spend
FROM per_company p
JOIN b2b_ecommerce.gold.dim_companies c
  ON c.company_id = p.company_id
 AND c.__END_AT IS NULL

In [0]:
%sql
-- Verification: shape, totals, cohort integrity. Expected: 390 rows = 390 distinct
-- companies = 26 cohorts summing to 390 companies, SUM(clv_net) = 28,333,498.48 and
-- SUM(clv_gross) = 30,723,269.53 (exact KPI 1/2/3/4 reconciliation), cohort_year_month
-- within 202410..202611, active_months <= lifetime_months for every company,
-- 0 null names/countries/industries/clv, first_order_date <= last_order_date,
-- avg_monthly_net_spend consistent with clv_net / active_months.
SELECT
  COUNT(*)                                            AS rows_companies,
  COUNT(DISTINCT company_id)                          AS distinct_companies,
  COUNT(DISTINCT cohort_year_month)                   AS cohorts,
  MIN(cohort_year_month)                               AS first_cohort,
  MAX(cohort_year_month)                               AS last_cohort,
  SUM(clv_net)                                        AS sum_clv_net,
  SUM(clv_gross)                                      AS sum_clv_gross,
  SUM(order_count)                                    AS total_orders,
  SUM(CASE WHEN active_months > lifetime_months THEN 1 ELSE 0 END) AS active_gt_lifetime,
  SUM(CASE WHEN first_order_date > last_order_date THEN 1 ELSE 0 END) AS first_after_last,
  COUNT(DISTINCT CASE WHEN company_name IS NULL OR country IS NULL
                       OR industry IS NULL OR clv_net IS NULL THEN company_id END) AS companies_with_nulls,
  SUM(CASE WHEN ABS(avg_monthly_net_spend - ROUND(clv_net / NULLIF(active_months, 0), 2)) > 0.005
           THEN 1 ELSE 0 END)                          AS bad_monthly_spend
FROM b2b_ecommerce.gold.kpi_5_customer_lifetime_value

In [0]:
%sql
-- Verification: independently recompute every company straight from the raw fact
-- keyed on company_sk (no dim join for the measures, no shared CTEs with the view),
-- then compare against the view on cohort assignment, both CLV bases, order count,
-- active months and lifetime months. Expected: companies_checked = 390 and every
-- match count = 390 (a different path than the view's company_id aggregation;
-- identical today because company_id <-> company_sk is 1:1 in the fact).
WITH recompute AS (
  SELECT
    company_sk,
    MIN(order_date)                                                          AS first_order_date,
    MAX(order_date)                                                          AS last_order_date,
    COUNT(DISTINCT order_id)                                                 AS order_count,
    COUNT(DISTINCT DATE_FORMAT(order_date, 'yyyyMM'))                        AS active_months,
    SUM(line_total)                                                          AS clv_gross,
    SUM(CASE WHEN status <> 'cancelled' THEN line_total ELSE 0 END)          AS clv_net
  FROM b2b_ecommerce.gold.fact_order_items
  WHERE order_date_valid = true
  GROUP BY company_sk
),
bridge AS (
  SELECT company_sk, company_id
  FROM b2b_ecommerce.gold.dim_companies
  GROUP BY company_sk, company_id
),
chk AS (
  SELECT
    b.company_id,
    r.first_order_date,
    r.last_order_date,
    r.order_count,
    r.active_months,
    ((YEAR(r.last_order_date) - YEAR(r.first_order_date)) * 12
       + (MONTH(r.last_order_date) - MONTH(r.first_order_date)) + 1) AS lifetime_months,
    r.clv_gross,
    r.clv_net
  FROM recompute r
  JOIN bridge b ON b.company_sk = r.company_sk
)
SELECT
  COUNT(*) AS companies_checked,
  SUM(CASE WHEN v.cohort_year_month = CAST(DATE_FORMAT(c.first_order_date, 'yyyyMM') AS INT) THEN 1 ELSE 0 END) AS cohort_match,
  SUM(CASE WHEN v.clv_net = c.clv_net THEN 1 ELSE 0 END)      AS clv_net_match,
  SUM(CASE WHEN v.clv_gross = c.clv_gross THEN 1 ELSE 0 END) AS clv_gross_match,
  SUM(CASE WHEN v.order_count = c.order_count THEN 1 ELSE 0 END) AS orders_match,
  SUM(CASE WHEN v.active_months = c.active_months THEN 1 ELSE 0 END) AS active_months_match,
  SUM(CASE WHEN v.lifetime_months = c.lifetime_months THEN 1 ELSE 0 END) AS lifetime_match
FROM b2b_ecommerce.gold.kpi_5_customer_lifetime_value v
JOIN chk c ON c.company_id = v.company_id

In [0]:
%sql
-- Cohort segmentation: one row per acquisition cohort (monthly grain), ordered
-- chronologically. Cohort-level CLV, average CLV per company in the cohort, and
-- average active months (early cohorts should show longer observed lifetimes).
SELECT
  cohort_year_month,
  cohort_quarter,
  COUNT(*)                                            AS companies,
  SUM(order_count)                                    AS orders,
  SUM(clv_net)                                        AS cohort_clv_net,
  ROUND(AVG(clv_net), 2)                              AS avg_clv_net_per_company,
  ROUND(100 * SUM(clv_net) / SUM(SUM(clv_net)) OVER (), 2) AS cohort_clv_share_pct,
  ROUND(AVG(active_months), 2)                        AS avg_active_months,
  ROUND(AVG(lifetime_months), 2)                     AS avg_lifetime_months,
  ROUND(AVG(avg_monthly_net_spend), 2)               AS avg_monthly_spend_per_company
FROM b2b_ecommerce.gold.kpi_5_customer_lifetime_value
GROUP BY cohort_year_month, cohort_quarter
ORDER BY cohort_year_month

In [0]:
%sql
-- Highest-value buyer companies (net CLV), with their acquisition cohort and
-- spend pattern.
SELECT company_id, company_name, country, industry,
       cohort_year_month, first_order_date, last_order_date,
       order_count, active_months, lifetime_months,
       clv_gross, clv_net, avg_monthly_net_spend
FROM b2b_ecommerce.gold.kpi_5_customer_lifetime_value
ORDER BY clv_net DESC
LIMIT 10

In [0]:
%sql
-- Coarser quarterly segmentation (requirements: "segment into cohorts") -
-- companies, CLV and average CLV per cohort quarter. Useful executive view of
-- acquisition waves.
SELECT
  cohort_quarter,
  COUNT(*)                              AS companies,
  SUM(clv_net)                          AS clv_net,
  ROUND(AVG(clv_net), 2)                AS avg_clv_net_per_company,
  ROUND(100 * SUM(clv_net) / SUM(SUM(clv_net)) OVER (), 2) AS clv_share_pct,
  ROUND(AVG(active_months), 2)          AS avg_active_months
FROM b2b_ecommerce.gold.kpi_5_customer_lifetime_value
GROUP BY cohort_quarter
ORDER BY cohort_quarter